# Hands-on 38 (Bab 15): Proyek Akhir End-to-End

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Capstone: simulator -> MQTT -> subscriber Python (validasi payload, preprocessing,
threshold baseline + Isolation Forest) -> storage -> alert. Struktur skrip mengikuti
buku: `scripts/publishers/`, `scripts/inference/`, `scripts/subscribers/`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Latih Model Baseline Isolation Forest

Baseline dilatih pada rentang operasi normal deployment (rentang payload normal
publisher); formula model persis buku. Lihat docstring
`scripts/inference/train_capstone_iforest.py` untuk alasannya.

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/inference/train_capstone_iforest.py"], capture_output=True, text=True)
print(r.stdout)

[SUKSES] Model disimpan ke: data/models/capstone_iforest.joblib



## 2. Jalankan Sistem (broker + subscriber+alert + listener + publisher)

In [3]:
influx_hapus("capstone_device_monitoring"); influx_hapus("capstone_alerts")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/subscribers/capstone_subscriber_with_alert.py", "capstone_sub_alert_nb.log")
    listener = latar("scripts/capstone_alert_listener.py", "capstone_listener_nb.log", ("--max-messages", "1", "--timeout", "60"))
    time.sleep(2)
    pub = latar("scripts/publishers/capstone_publisher.py", "capstone_pub_nb.log")
    time.sleep(40)   # sekitar 20 pesan (interval 2 detik)
    henti(pub, sub, listener)
    print("--- Subscriber + alert ---")
    print(log_teks("capstone_sub_alert_nb.log"))
    print("--- Listener topik alert ---")
    print(log_teks("capstone_listener_nb.log")[:500])
finally:
    broker.terminate()

--- Subscriber + alert ---
Subscribed to capstone/iotnode01/telemetry
ALERT: {'timestamp': '2026-10-22T13:46:30', 'device_id': 'iotnode01', 'temperature_c': 24.68, 'humidity_pct': 69.27, 'power_watt': 258.19, 'current_amp': 8.23, 'reason': 'iforest'}
ALERT: {'timestamp': '2026-10-22T13:47:02', 'device_id': 'iotnode01', 'temperature_c': 26.38, 'humidity_pct': 42.88, 'power_watt': 90.72, 'current_amp': 9.2, 'reason': 'iforest'}

--- Listener topik alert ---
[INFO] Menunggu alert di topik: capstone/iotnode01/alert

=== NOTIFIKASI ALERT ===
Waktu   : 2026-10-22T13:46:30
Device  : iotnode01
Alasan  : iforest
Daya    : 258.19 W
Arus    : 8.23 A

[SUKSES] 1 notifikasi disimpan ke outputs/reports/capstone_notifications.csv



## 3. Verifikasi Storage

In [4]:
mon = influx_df("capstone_device_monitoring")
alerts = influx_df("capstone_alerts")
print(mon.head(8).to_string(index=False))
print(f"\nAnomali: {mon['iforest_anomaly'].sum()}, Threshold: {mon['threshold_alert'].sum()}, Alert: {len(alerts)}")

                    _time device_id  current_amp  humidity_pct  iforest_anomaly  power_watt  temperature_c  threshold_alert
2026-10-22 13:46:30+00:00 iotnode01         8.23         69.27                1      258.19          24.68                0
2026-10-22 13:46:32+00:00 iotnode01         8.27         50.72                0       99.77          28.02                0
2026-10-22 13:46:34+00:00 iotnode01         8.87         62.05                0      164.95          29.08                0
2026-10-22 13:46:36+00:00 iotnode01         8.69         55.38                0      216.83          27.93                0
2026-10-22 13:46:38+00:00 iotnode01         7.60         63.81                0      113.06          25.48                0
2026-10-22 13:46:40+00:00 iotnode01         7.06         47.27                0      103.90          31.82                0
2026-10-22 13:46:42+00:00 iotnode01         8.97         63.84                0       94.03          25.77                0
2026-10-

## Penutup

Sistem capstone berjalan ujung ke ujung dan alert sampai ke topik terpisah. Baseline
training memakai rentang operasi normal deployment (rentang payload normal publisher),
sesuai prinsip Bab 13 bahwa data training harus mewakili kondisi deployment - formula
model tetap persis buku (IsolationForest, n_estimators=200, contamination=0.05).
Hasilnya alert turun dari 14 menjadi 3 per 15 pesan: injeksi anomali tertangkap
(threshold + iforest), plus flag batas sesekali yang wajar untuk contamination 5% -
"notifikasi harus cukup berguna, bukan terlalu berisik" (Bab 15). Untuk deployment
penuh, aktifkan `--influx` dan susun dashboard Grafana (panel suhu, kelembaban, daya,
arus, iforest_anomaly, threshold_alert, histori, dan jumlah alert per jam).

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 40 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 10. Verifikasi sistem end-to-end

Periksa hal berikut:

publisher mengirim payload,
broker menerima,
subscriber mencetak prediksi,
InfluxDB menerima data,
Grafana menampilkan histori dan status.

Jika semuanya berjalan, maka capstone end-to-end dasar sudah hidup.

## Jawaban dari data: Verifikasi end-to-end (Hands-on 38): jawaban dari hasil uji ini

In [5]:
import json
print("Publisher mengirim payload: lihat log publisher di atas (outputs/logs/capstone_pub_nb.log)")
mon = influx_df("capstone_device_monitoring")
print(f"Broker menerima dan subscriber mencetak prediksi: {len(mon)} baris tercatat dari subscriber")
print(f"InfluxDB menerima data: {len(mon)} baris pada measurement capstone_device_monitoring")
print("Grafana: diuji terpisah (datasource sehat, 8 panel mengembalikan data); lihat README.")

Publisher mengirim payload: lihat log publisher di atas (outputs/logs/capstone_pub_nb.log)
Broker menerima dan subscriber mencetak prediksi: 20 baris tercatat dari subscriber
InfluxDB menerima data: 20 baris pada measurement capstone_device_monitoring
Grafana: diuji terpisah (datasource sehat, 8 panel mengembalikan data); lihat README.
